In [1]:
# [CELL 1] CONFIGURATION & PARAMETERS
from pathlib import Path
from collections import Counter
import csv, json, shutil

LAST_NAME = "Ongjoco"
STUDENT_ID = "TUPM-26-1852"
SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_YEAR = str(2015 + SEED_DIGIT)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
print(f"Target Year: {TARGET_YEAR}")

Target Year: 2017


In [2]:
# [CELL 2] PROCESS AND ANALYZE INCIDENTS
total_records = 0
target_incidents = []
attack_counts = Counter()
total_loss = 0.0

if dataset_path.exists():
    with open(dataset_path, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total_records += 1
            # Adjust column names based on exact dataset headers if necessary (e.g., Year, Attack Type, Financial Loss)
            year_val = row.get("Year", "").strip()
            if year_val == TARGET_YEAR:
                attack_type = row.get("Attack Type", "Unknown").strip()
                attack_counts[attack_type] += 1
                
                try:
                    loss = float(row.get("Financial Loss (in Million $)", 0))
                except ValueError:
                    loss = 0.0
                
                total_loss += loss
                target_incidents.append({
                    "date": row.get("Date", ""),
                    "attack_type": attack_type,
                    "financial_loss": loss,
                    "description": row.get("Description", "")
                })

count_target = len(target_incidents)
avg_loss = total_loss / count_target if count_target > 0 else 0.0

# Sort by financial loss descending
target_incidents.sort(key=lambda x: x["financial_loss"], reverse=True)
top_5_costliest = target_incidents[:5]

print("Total Records in Dataset:", total_records)
print("Incidents in Target Year:", count_target)
print("Total Financial Loss: $", round(total_loss, 2), "M")
print("Average Financial Loss: $", round(avg_loss, 2), "M")


Total Records in Dataset: 0
Incidents in Target Year: 0
Total Financial Loss: $ 0.0 M
Average Financial Loss: $ 0.0 M


In [4]:
# [CELL 3] EXPORT REPORTS, JSON & BACKUP
csv_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_filtered.csv"
json_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_top5.json"
report_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cybersecurity_report.txt"

with open(csv_out, "w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=["date", "attack_type", "financial_loss", "description"])
    writer.writeheader()
    writer.writerows(target_incidents)

with open(json_out, "w", encoding="utf-8") as j:
    json.dump(top_5_costliest, j, indent=4)

report_content = f"""CYBERSECURITY INCIDENT REPORT ({TARGET_YEAR})
Student: {LAST_NAME} ({STUDENT_ID})
----------------------------------------
Total Records in Dataset: {total_records}
Incidents in Target Year: {count_target}
Total Financial Loss: ${total_loss:.2f} Million
Average Financial Loss: ${avg_loss:.2f} Million

Attack Type Breakdown:
{dict(attack_counts)}
----------------------------------------
"""
with open(report_out, "w", encoding="utf-8") as r:
    r.write(report_content)

backup_json = BACKUP_DIR / json_out.name
shutil.copy2(json_out, backup_json)

print("Report created:", report_out)
print("JSON Backup Exists:", backup_json.exists(), "| Size:", backup_json.stat().st_size, "bytes")


Report created: c:\Users\U S E R - P C\Lab6_File_Handling_Ongjoco\Lab6_File_Handling_Additional_Exercises\output\Ongjoco_TUPM-26-1852_cybersecurity_report.txt
JSON Backup Exists: True | Size: 2 bytes
